In [1]:
import pandas as pd
import numpy as np
import joblib
import time
import os

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded.")

Libraries loaded.


In [2]:
X_train = pd.read_pickle("../../data/processed/X_train.pkl")
X_test = pd.read_pickle("../../data/processed/X_test.pkl")
y_train = pd.read_pickle("../../data/processed/y_train.pkl")
y_test = pd.read_pickle("../../data/processed/y_test.pkl")

print("Data loaded.")
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Data loaded.
X_train: (472432, 421)
X_test : (118108, 421)


In [ ]:
import optuna
from xgboost import XGBClassifier

print("Optuna and XGBoost loaded.")

In [ ]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 900),
        "max_depth": trial.suggest_int("max_depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 10),
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1,
    }

    model_xgb = XGBClassifier(**params)

    model_xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,
    )

    y_prob = model_xgb.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, y_prob)


study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=15)

print("Best XGBoost ROC-AUC:", study_xgb.best_value)
print("Best parameters:")
print(study_xgb.best_params)

In [ ]:
best_xgb_params = study_xgb.best_params.copy()

best_xgb_params.update({
    "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
})

xgb_tuned_model = XGBClassifier(**best_xgb_params)

start = time.time()
xgb_tuned_model.fit(X_train, y_train)
training_time = round(time.time() - start, 2)

y_prob_xgb = xgb_tuned_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

xgb_tuned_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": (y_pred_xgb == y_test).mean(),
    "Precision": precision_score(y_test, y_pred_xgb),
    "Recall": recall_score(y_test, y_pred_xgb),
    "F1 Score": f1_score(y_test, y_pred_xgb),
    "ROC-AUC": roc_auc_score(y_test, y_prob_xgb),
    "PR-AUC": average_precision_score(y_test, y_prob_xgb),
    "Training Time (s)": training_time,
}

print(xgb_tuned_results)
print("\nClassification Report")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_xgb))

In [ ]:
os.makedirs("../../ml/models", exist_ok=True)

joblib.dump(xgb_tuned_model, "../../ml/models/xgboost_tuned.pkl")
joblib.dump(study_xgb, "../../ml/models/xgboost_optuna_study.pkl")

print("Tuned XGBoost model and study saved.")

In [ ]:
import optuna
from catboost import CatBoostClassifier

print("Optuna and CatBoost loaded.")

In [ ]:
def objective_cat(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 300, 900),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "border_count": trial.suggest_int("border_count", 32, 255),
        "random_strength": trial.suggest_float("random_strength", 0, 5),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 5),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "auto_class_weights": "Balanced",
        "random_seed": 42,
        "verbose": False,
        "thread_count": -1,
    }

    model_cat = CatBoostClassifier(**params)

    model_cat.fit(
        X_train,
        y_train,
        eval_set=(X_test, y_test),
        use_best_model=True,
        verbose=False,
    )

    y_prob = model_cat.predict_proba(X_test)[:, 1]
    return roc_auc_score(y_test, y_prob)


study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=15)

print("Best CatBoost ROC-AUC:", study_cat.best_value)
print("Best parameters:")
print(study_cat.best_params)

In [ ]:
best_cat_params = study_cat.best_params.copy()

best_cat_params.update({
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "random_seed": 42,
    "verbose": False,
    "thread_count": -1,
})

cat_tuned_model = CatBoostClassifier(**best_cat_params)

start = time.time()
cat_tuned_model.fit(X_train, y_train)
training_time = round(time.time() - start, 2)

y_prob_cat = cat_tuned_model.predict_proba(X_test)[:, 1]
y_pred_cat = (y_prob_cat >= 0.5).astype(int)

cat_tuned_results = {
    "Model": "CatBoost Tuned",
    "Accuracy": (y_pred_cat == y_test).mean(),
    "Precision": precision_score(y_test, y_pred_cat),
    "Recall": recall_score(y_test, y_pred_cat),
    "F1 Score": f1_score(y_test, y_pred_cat),
    "ROC-AUC": roc_auc_score(y_test, y_prob_cat),
    "PR-AUC": average_precision_score(y_test, y_prob_cat),
    "Training Time (s)": training_time,
}

print(cat_tuned_results)
print("\nClassification Report")
print(classification_report(y_test, y_pred_cat))
print("\nConfusion Matrix")
print(confusion_matrix(y_test, y_pred_cat))

In [ ]:
joblib.dump(cat_tuned_model, "../../ml/models/catboost_tuned.pkl")
joblib.dump(study_cat, "../../ml/models/catboost_optuna_study.pkl")

print("Tuned CatBoost model and study saved.")

In [5]:
import pandas as pd
import numpy as np
import joblib
import os
import json

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    brier_score_loss
)

print("Recovery libraries loaded.")

Recovery libraries loaded.


In [6]:
X_test = pd.read_pickle("../../data/processed/X_test.pkl")
y_test = pd.read_pickle("../../data/processed/y_test.pkl")

lightgbm_model = joblib.load("../../ml/models/lightgbm_tuned.pkl")
xgb_tuned_model = joblib.load("../../ml/models/xgboost_tuned.pkl")
cat_tuned_model = joblib.load("../../ml/models/catboost_tuned.pkl")

print("Saved models loaded successfully.")

Saved models loaded successfully.


In [7]:
isotonic_calibrator = joblib.load("../../ml/registry/champion_isotonic_calibrator.pkl")

y_prob_iso = isotonic_calibrator.predict_proba(X_test)[:, 1]

print("Isotonic calibrator loaded.")
print("ROC-AUC:", roc_auc_score(y_test, y_prob_iso))
print("PR-AUC:", average_precision_score(y_test, y_prob_iso))
print("Brier:", brier_score_loss(y_test, y_prob_iso))

Isotonic calibrator loaded.
ROC-AUC: 0.9762128826095815
PR-AUC: 0.8766581652359666
Brier: 0.008616176835254724


In [8]:
import json
import os
import pandas as pd
import joblib

os.makedirs("../../ml/registry", exist_ok=True)

model_registry = pd.DataFrame([
    {
        "model_name": "XGBoost Tuned",
        "model_version": "xgboost-tuned-v1",
        "roc_auc": 0.973874,
        "pr_auc": 0.848838,
        "precision": 0.722751,
        "recall": 0.808614,
        "f1_score": 0.763275,
        "training_time_seconds": 268.49,
        "role": "challenger",
    },
    {
        "model_name": "CatBoost Tuned",
        "model_version": "catboost-tuned-v1",
        "roc_auc": 0.967136,
        "pr_auc": 0.812986,
        "precision": 0.566394,
        "recall": 0.820469,
        "f1_score": 0.670158,
        "training_time_seconds": 1155.63,
        "role": "challenger",
    },
    {
        "model_name": "LightGBM Tuned + Isotonic",
        "model_version": "lightgbm-tuned-v2-calibrated",
        "roc_auc": 0.976213,
        "pr_auc": 0.876658,
        "precision": 0.882700,
        "recall": 0.797484,
        "f1_score": 0.837931,
        "brier_score": 0.008616,
        "training_time_seconds": None,
        "role": "champion",
    },
])

model_registry.to_csv("../../ml/registry/model_registry.csv", index=False)

champion_card = {
    "champion_model": "LightGBM Tuned + Isotonic",
    "model_version": "lightgbm-tuned-v2-calibrated",
    "status": "production_ready",
    "calibration": "isotonic",
    "roc_auc": 0.976213,
    "pr_auc": 0.876658,
    "brier_score": 0.008616,
    "recommended_threshold_policy": {
        "low_risk": "score < 0.40",
        "manual_review": "0.40 <= score < 0.60",
        "high_alert": "score >= 0.60",
    },
    "champion_reason": "Selected because it achieved the best ROC-AUC, strongest calibrated probability reliability, best F1 balance, and lowest false-positive burden compared with tuned XGBoost and CatBoost.",
    "challengers": ["XGBoost Tuned", "CatBoost Tuned"],
}

with open("../../ml/registry/champion_model_card.json", "w") as f:
    json.dump(champion_card, f, indent=4)

joblib.dump(lightgbm_model, "../../ml/registry/champion_lightgbm_model.pkl")
joblib.dump(isotonic_calibrator, "../../ml/registry/champion_isotonic_calibrator.pkl")

print("Model registry created successfully.")
model_registry.sort_values("roc_auc", ascending=False)

Model registry created successfully.


,model_name,model_version,roc_auc,pr_auc,precision,recall,f1_score,training_time_seconds,role,brier_score
2,LightGBM Tuned + Isotonic,lightgbm-tuned-v2-calibrated,0.976213,0.876658,0.882700,0.797484,0.837931,NaN,champion,0.008616
0,XGBoost Tuned,xgboost-tuned-v1,0.973874,0.848838,0.722751,0.808614,0.763275,268.49,challenger,NaN
1,CatBoost Tuned,catboost-tuned-v1,0.967136,0.812986,0.566394,0.820469,0.670158,1155.63,challenger,NaN
